
# 05_Create_Document_Chunks

Crea fragmentos de texto **section-aware** para recuperación léxica,vectorial e híbrida.

Tareas:

1. Procesa únicamente documentos con `chunking_status` pendiente o fallido;
2. Usa solo las secciones de la `cleaning_run_id` vigente;
3. Excluye referencias y secciones vacías;
4. Nunca cruza límites de sección;
5. Reemplaza autoritativamente los chunks de cada documento procesado;
6. Elimina metadatos duplicados del documento en la tabla de chunks;
7. Elimina `page_start/page_end`, porque no existe un mapeo fiable sección ↔ página;
8. Añade `content_hash` para detectar cambios de contenido al publicar en Supabase;
9. Actualiza `chunking_status` en `pmc_inventory`;
10. Mantiene trazabilidad por `cleaning_run_id` y `chunking_run_id`.


In [0]:
from __future__ import annotations

import hashlib
import json
import uuid

from datetime import datetime, timezone
from typing import Any

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:

# ============================================================
# Configuración
# ============================================================

CATALOG_NAME = "workspace"
SCHEMA_NAME = "tfm_pmc"

INVENTORY_TABLE = (
    f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_inventory"
)

CLEAN_SECTION_TABLE = (
    f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_clean_sections"
)

CLEAN_DOCUMENT_TABLE = (
    f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_clean_documents"
)

CHUNK_TABLE = (
    f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_document_chunks"
)

PIPELINE_RUNS_TABLE = (
    f"{CATALOG_NAME}.{SCHEMA_NAME}.pipeline_runs"
)

PIPELINE_NAME = "05_Create_Document_Chunks_v3_Clean"

MAX_DOCUMENTS = 20

# Mantenemos exactamente los parámetros de la versión latest.
CHUNK_SIZE_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

# Una sección completa de al menos 20 palabras se conserva aunque sea
# menor que el tamaño objetivo.
MIN_SECTION_WORDS = 20

CHUNKING_METHOD = (
    "section_word_window_220_overlap_40_v3"
)

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

print(f"Run ID: {RUN_ID}")
print(f"Chunk size: {CHUNK_SIZE_WORDS} words")
print(f"Overlap: {CHUNK_OVERLAP_WORDS} words")
print(f"Minimum section: {MIN_SECTION_WORDS} words")
print(f"Target table: {CHUNK_TABLE}")


Run ID: fcfabe4d-a5c0-4cf9-bcaa-b347cd7b05d6
Chunk size: 220 words
Overlap: 40 words
Minimum section: 20 words
Target table: workspace.tfm_pmc.pmc_document_chunks


In [0]:

# ============================================================
# Crear tabla de chunks v3
# ============================================================
#
# En este ciclo controlado venimos de un reset del schema, por lo que
# la tabla se crea directamente con el esquema final simplificado.

spark.sql(
    f'''
    CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
        pmcid STRING NOT NULL,
        article_version STRING NOT NULL,

        chunk_index INT NOT NULL,
        section_chunk_index INT NOT NULL,

        section_index INT NOT NULL,
        section_id STRING,
        parent_section_id STRING,

        section_type STRING,
        section_title STRING,
        section_path STRING,
        section_level INT,

        chunk_text STRING NOT NULL,
        word_count INT NOT NULL,
        character_count BIGINT NOT NULL,
        content_hash STRING NOT NULL,

        chunking_method STRING NOT NULL,

        cleaning_run_id STRING NOT NULL,
        chunking_run_id STRING NOT NULL,

        processed_at TIMESTAMP NOT NULL,
        updated_at TIMESTAMP NOT NULL
    )
    USING DELTA
    '''
)

print("Chunk table is ready.")


Chunk table is ready.


In [0]:

# ============================================================
# Seleccionar documentos pendientes
# ============================================================

clean_documents_df = (
    spark.table(CLEAN_DOCUMENT_TABLE)
    .filter(
        F.col("cleaning_status").isin(
            "completed",
            "completed_with_errors",
        )
    )
    .select(
        "pmcid",
        "article_version",
        "cleaning_run_id",
    )
)

inventory_status_df = (
    spark.table(INVENTORY_TABLE)
    .filter(
        F.col("is_latest_version") == True
    )
    .select(
        "pmcid",
        "article_version",
        "chunking_status",
    )
)

pending_documents_df = (
    clean_documents_df.alias("clean")
    .join(
        inventory_status_df.alias("inventory"),
        on=[
            "pmcid",
            "article_version",
        ],
        how="inner",
    )
    .filter(
        F.coalesce(
            F.col("inventory.chunking_status"),
            F.lit("pending"),
        ).isin(
            "pending",
            "failed",
        )
    )
    .select(
        "pmcid",
        "article_version",
        F.col("clean.cleaning_run_id").alias(
            "cleaning_run_id"
        ),
    )
    .orderBy(
        "pmcid",
        "article_version",
    )
    .limit(MAX_DOCUMENTS)
)

documents_selected = pending_documents_df.count()

print(
    "Documents selected:",
    documents_selected,
)

if documents_selected == 0:
    dbutils.notebook.exit(
        "No pending documents to chunk."
    )

display(
    pending_documents_df
)


Documents selected: 20


pmcid,article_version,cleaning_run_id
PMC13538227,PMC13538227.1,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13538279,PMC13538279.1,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13539412,PMC13539412.1,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13540131,PMC13540131.2,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13543812,PMC13543812.1,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13544149,PMC13544149.1,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13545448,PMC13545448.1,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13547081,PMC13547081.1,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13547469,PMC13547469.1,460d5cd5-95d3-495d-ba67-21bdceb792c2
PMC13552502,PMC13552502.1,460d5cd5-95d3-495d-ba67-21bdceb792c2


In [0]:

# ============================================================
# Recuperar únicamente secciones elegibles para retrieval
# ============================================================

selected_keys_df = (
    pending_documents_df
    .select(
        "pmcid",
        "article_version",
        "cleaning_run_id",
    )
)

sections_df = (
    spark.table(CLEAN_SECTION_TABLE).alias("sections")
    .join(
        selected_keys_df.alias("selected"),
        on=[
            "pmcid",
            "article_version",
            "cleaning_run_id",
        ],
        how="inner",
    )
    .filter(
        F.col(
            "sections.cleaning_status"
        ) == "completed"
    )
    .filter(
        F.col(
            "sections.include_in_retrieval"
        ) == True
    )
    .filter(
        F.col(
            "sections.is_reference_section"
        ) == False
    )
    .filter(
        F.length(
            F.trim(
                F.col(
                    "sections.clean_content"
                )
            )
        ) > 0
    )
    .select(
        "pmcid",
        "article_version",

        "section_index",
        "section_id",
        "parent_section_id",

        "section_type",
        "section_title",
        "section_path",
        "section_level",

        "clean_content",
        "cleaning_run_id",
    )
    .orderBy(
        "pmcid",
        "article_version",
        "section_index",
    )
)

sections_selected = sections_df.count()

print(
    "Retrievable sections selected:",
    sections_selected,
)

reference_leak_count = (
    spark.table(CLEAN_SECTION_TABLE)
    .alias("sections")
    .join(
        selected_keys_df.alias("selected"),
        on=[
            "pmcid",
            "article_version",
            "cleaning_run_id",
        ],
        how="inner",
    )
    .filter(
        F.col(
            "sections.include_in_retrieval"
        ) == True
    )
    .filter(
        F.col(
            "sections.is_reference_section"
        ) == True
    )
    .count()
)

print(
    "Reference sections leaking into retrieval:",
    reference_leak_count,
)

if reference_leak_count != 0:
    raise RuntimeError(
        "Reference sections are incorrectly marked for retrieval."
    )

display(
    sections_df.limit(50)
)


Retrievable sections selected: 342
Reference sections leaking into retrieval: 0


pmcid article_version section_index section_id parent_section_id section_type section_title section_path section_level clean_content cleaning_run_id PMC13538227 PMC13538227.1 0 abstract null abstract Abstract Abstract 0 Introduction Whether spontaneous cervical artery dissection (sCeAD), the leading cause of ischemic stroke in young adults, represents the manifestation of unrecognized hereditary connective tissue disorders (HCTDs) and whether HCTDs have a major impact in the epidemiology of the disease is a matter of ongoing debate. We aimed at determining the frequency of clinically relevant genetic variants (CRGVs) in a cohort of unselected sCeAD patients by targeted next-generation sequencing (NGS) approach. Methods We designed a high-throughput sequencing panel to identify variants in 38 candidate genes associated with arterial dissection or aneurysm and screened patients with apparently sporadic sCeAD, consecutively referred to one comprehensive stroke center from August 2020 to December 2025. The frequency of known disease-causing and pertinent variants of uncertain significance (VUS) was calculated. Then, we performed a systematic review of all studies evaluating the prevalence of monogenic disorders among sCeAD patients up to December 2025. Results Among 183 patients (males, 51.3%; mean age, 42.0 ± 11.3 years), 2 (1.1%) carried a CeAD-causing variant in COL3A1 ( NM_000090.4:c.2959G > A:p.Gly987Ser) and ABCC6 ( NM_001351800.1:c.3071G > A:p.Arg1024Gln), respectively. In addition, we identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of CeAD. Conclusion Systematic search for rare disease-causing variants should not be recommended in all sCeAD cases but it should be limited to selected individuals with a high pre-test probability to harbor a monogenic disease. Supplementary Information The online version contains supplementary material available at https://doi.org/10.1007/s10072-026-09308-6. 460d5cd5-95d3-495d-ba67-21bdceb792c2 PMC13538227 PMC13538227.1 1 Sec1 null section Introduction Introduction 1 Arterial dissection is, by definition, the accumulation of blood within the wall of an artery. Once considered uncommon, dissections of carotid or vertebral arteries (cervical artery dissections, CeADs) are now recognized as the major cause of ischemic stroke in young adults, accounting for approximately 20% of cases in patients under 45 years of age. Notwithstanding, the pathogenesis of CeAD is still poorly defined, especially in those cases that occur spontaneously (spontaneous CeAD, sCeAD), without any identifiable precipitating events [ 1 ]. Several arguments point toward a key role played by the connective tissue component of the arterial wall. The finding of composite collagen fibrils and fragmented elastic fibers on electron microscopic examination of skin biopsy specimens in more than half of patients with sCeAD [ 2 – 4 ] and the identification of clinically detectable signs of connective tissue aberration in most sCeAD patients support the hypothesis of a generalized structural connective tissue defect predisposing to disease occurrence, even in sporadic cases without evident signs of a known hereditary connective tissue disorder (HCTD) [ 5, 6 ]. In this view, the reduced or altered biosynthesis of the extracellular matrix (ECM) components might lead to a generalized structural weakness of the vessel wall and explain the increased risk for sCeAD. Whether such arteriopathy underlying sCeAD represents a mild phenotypic manifestation of HCTD has never been fully elucidated. Most of the studies so far have been conducted on small series with limited number of screened genes [ 7, 8 ]. Recently, next-generation sequencing (NGS) has become an effective tool to detect rare monogenic disorders. However, only a few NGS studies

In [0]:
# ============================================================
# Construcción de chunks dentro de una sección
# ============================================================

def create_section_chunks(
    text: str | None,
    chunk_size: int = CHUNK_SIZE_WORDS,
    overlap: int = CHUNK_OVERLAP_WORDS,
    min_section_words: int = MIN_SECTION_WORDS,
) -> list[dict[str, Any]]:
    '''
    Crea ventanas de palabras sin cruzar el límite de la sección.

    Reglas:
    - secciones menores a MIN_SECTION_WORDS se omiten;
    - una sección <= CHUNK_SIZE_WORDS genera un solo chunk;
    - una sección larga genera ventanas con overlap;
    - el último fragmento se conserva si tiene >= MIN_SECTION_WORDS.
    '''

    if chunk_size <= 0:
        raise ValueError(
            "chunk_size must be greater than zero."
        )

    if overlap < 0:
        raise ValueError(
            "overlap cannot be negative."
        )

    if overlap >= chunk_size:
        raise ValueError(
            "overlap must be smaller than chunk_size."
        )

    if not text:
        return []

    words = text.split()

    if len(words) < min_section_words:
        return []

    if len(words) <= chunk_size:
        chunk_text = " ".join(words).strip()

        return [{
            "section_chunk_index": 0,
            "chunk_text": chunk_text,
            "word_count": len(words),
            "character_count": len(chunk_text),
        }]

    chunks = []

    step = chunk_size - overlap
    start = 0
    section_chunk_index = 0

    while start < len(words):
        end = min(
            start + chunk_size,
            len(words),
        )

        window = words[start:end]

        if not window:
            break

        # No crear un residual diminuto al final de una sección.
        if (
            len(window) < min_section_words
            and chunks
        ):
            break

        chunk_text = " ".join(
            window
        ).strip()

        chunks.append({
            "section_chunk_index":
                section_chunk_index,
            "chunk_text":
                chunk_text,
            "word_count":
                len(window),
            "character_count":
                len(chunk_text),
        })

        section_chunk_index += 1

        if end >= len(words):
            break

        start += step

    return chunks

In [0]:

# ============================================================
# Indexar secciones por documento
# ============================================================
#
# Conservamos la estrategia serverless-safe de la versión latest.

sections_by_document: dict[
    tuple[str, str],
    list[dict[str, Any]]
] = {}

for record in sections_df.toLocalIterator():

    key = (
        record["pmcid"],
        record["article_version"],
    )

    sections_by_document.setdefault(
        key,
        [],
    ).append({
        "section_index":
            int(record["section_index"]),

        "section_id":
            record["section_id"],

        "parent_section_id":
            record["parent_section_id"],

        "section_type":
            record["section_type"],

        "section_title":
            record["section_title"],

        "section_path":
            record["section_path"],

        "section_level":
            (
                int(record["section_level"])
                if record["section_level"] is not None
                else None
            ),

        "clean_content":
            record["clean_content"],

        "cleaning_run_id":
            record["cleaning_run_id"],
    })

for key in sections_by_document:
    sections_by_document[key].sort(
        key=lambda section:
            section["section_index"]
    )

print(
    "Documents with retrievable sections:",
    len(sections_by_document),
)


Documents with retrievable sections: 20


In [0]:

# ============================================================
# Procesar documentos
# ============================================================

chunk_results: list[dict[str, Any]] = []
document_status_results: list[dict[str, Any]] = []

for index, document in enumerate(
    pending_documents_df.toLocalIterator(),
    start=1,
):
    pmcid = document["pmcid"]
    article_version = document["article_version"]
    current_cleaning_run_id = (
        document["cleaning_run_id"]
    )

    print(
        f"[{index}/{documents_selected}] "
        f"Chunking {article_version}"
    )

    processed_at = datetime.now(
        timezone.utc
    )

    try:
        sections = (
            sections_by_document.get(
                (
                    pmcid,
                    article_version,
                ),
                [],
            )
        )

        global_chunk_index = 0
        generated_for_document = 0

        for section in sections:

            # Defensa adicional contra mezcla de ejecuciones.
            if (
                section["cleaning_run_id"]
                != current_cleaning_run_id
            ):
                continue

            section_chunks = (
                create_section_chunks(
                    section["clean_content"]
                )
            )

            for chunk in section_chunks:

                chunk_text = chunk["chunk_text"]

                chunk_results.append({
                    "pmcid":
                        pmcid,

                    "article_version":
                        article_version,

                    "chunk_index":
                        global_chunk_index,

                    "section_chunk_index":
                        chunk[
                            "section_chunk_index"
                        ],

                    "section_index":
                        section["section_index"],

                    "section_id":
                        section["section_id"],

                    "parent_section_id":
                        section[
                            "parent_section_id"
                        ],

                    "section_type":
                        section["section_type"],

                    "section_title":
                        section["section_title"],

                    "section_path":
                        section["section_path"],

                    "section_level":
                        section["section_level"],

                    "chunk_text":
                        chunk_text,

                    "word_count":
                        int(chunk["word_count"]),

                    "character_count":
                        int(
                            chunk["character_count"]
                        ),

                    "content_hash":
                        hashlib.sha256(
                            chunk_text.encode("utf-8")
                        ).hexdigest(),

                    "chunking_method":
                        CHUNKING_METHOD,

                    "cleaning_run_id":
                        current_cleaning_run_id,

                    "chunking_run_id":
                        RUN_ID,

                    "processed_at":
                        processed_at,

                    "updated_at":
                        processed_at,
                })

                global_chunk_index += 1
                generated_for_document += 1

        if generated_for_document == 0:
            raise ValueError(
                "No valid retrievable chunks were generated."
            )

        document_status_results.append({
            "pmcid":
                pmcid,

            "article_version":
                article_version,

            "chunking_status":
                "completed",

            "chunk_count":
                generated_for_document,

            "error_message":
                None,

            "updated_at":
                processed_at,
        })

        print(
            "  Chunks generated:",
            generated_for_document,
        )

    except Exception as error:

        document_status_results.append({
            "pmcid":
                pmcid,

            "article_version":
                article_version,

            "chunking_status":
                "failed",

            "chunk_count":
                0,

            "error_message":
                str(error),

            "updated_at":
                processed_at,
        })

        print(
            "  FAILED:",
            str(error),
        )

print(
    "Total chunks generated:",
    len(chunk_results),
)


[1/20] Chunking PMC13538227.1
  Chunks generated: 25
[2/20] Chunking PMC13538279.1
  Chunks generated: 34
[3/20] Chunking PMC13539412.1
  Chunks generated: 43
[4/20] Chunking PMC13540131.2
  Chunks generated: 35
[5/20] Chunking PMC13543812.1
  Chunks generated: 38
[6/20] Chunking PMC13544149.1
  Chunks generated: 21
[7/20] Chunking PMC13545448.1
  Chunks generated: 48
[8/20] Chunking PMC13547081.1
  Chunks generated: 14
[9/20] Chunking PMC13547469.1
  Chunks generated: 24
[10/20] Chunking PMC13552502.1
  Chunks generated: 48
[11/20] Chunking PMC13552808.1
  Chunks generated: 11
[12/20] Chunking PMC13554935.1
  Chunks generated: 13
[13/20] Chunking PMC13559883.1
  Chunks generated: 49
[14/20] Chunking PMC13560980.1
  Chunks generated: 20
[15/20] Chunking PMC13561766.1
  Chunks generated: 27
[16/20] Chunking PMC13564229.1
  Chunks generated: 36
[17/20] Chunking PMC13564258.1
  Chunks generated: 5
[18/20] Chunking PMC13569294.1
  Chunks generated: 61
[19/20] Chunking PMC13570527.1
  Chunk

In [0]:

# ============================================================
# Esquemas Spark
# ============================================================

chunk_schema = spark.table(
    CHUNK_TABLE
).schema

status_schema = T.StructType([
    T.StructField(
        "pmcid",
        T.StringType(),
        False,
    ),
    T.StructField(
        "article_version",
        T.StringType(),
        False,
    ),
    T.StructField(
        "chunking_status",
        T.StringType(),
        False,
    ),
    T.StructField(
        "chunk_count",
        T.IntegerType(),
        False,
    ),
    T.StructField(
        "error_message",
        T.StringType(),
        True,
    ),
    T.StructField(
        "updated_at",
        T.TimestampType(),
        False,
    ),
])


In [0]:

# ============================================================
# Crear DataFrames
# ============================================================

chunks_df = None
document_status_df = None

if chunk_results:

    chunks_df = spark.createDataFrame(
        chunk_results,
        schema=chunk_schema,
    )

    display(
        chunks_df.limit(30)
    )

if document_status_results:

    document_status_df = (
        spark.createDataFrame(
            document_status_results,
            schema=status_schema,
        )
    )

    display(
        document_status_df
    )


pmcid,article_version,chunk_index,section_chunk_index,section_index,section_id,parent_section_id,section_type,section_title,section_path,section_level,chunk_text,word_count,character_count,content_hash,chunking_method,cleaning_run_id,chunking_run_id,processed_at,updated_at
PMC13538227,PMC13538227.1,0,0,0,abstract,null,abstract,Abstract,Abstract,0,"Introduction Whether spontaneous cervical artery dissection (sCeAD), the leading cause of ischemic stroke in young adults, represents the manifestation of unrecognized hereditary connective tissue disorders (HCTDs) and whether HCTDs have a major impact in the epidemiology of the disease is a matter of ongoing debate. We aimed at determining the frequency of clinically relevant genetic variants (CRGVs) in a cohort of unselected sCeAD patients by targeted next-generation sequencing (NGS) approach. Methods We designed a high-throughput sequencing panel to identify variants in 38 candidate genes associated with arterial dissection or aneurysm and screened patients with apparently sporadic sCeAD, consecutively referred to one comprehensive stroke center from August 2020 to December 2025. The frequency of known disease-causing and pertinent variants of uncertain significance (VUS) was calculated. Then, we performed a systematic review of all studies evaluating the prevalence of monogenic disorders among sCeAD patients up to December 2025. Results Among 183 patients (males, 51.3%; mean age, 42.0 ± 11.3 years), 2 (1.1%) carried a CeAD-causing variant in COL3A1 ( NM_000090.4:c.2959G > A:p.Gly987Ser) and ABCC6 ( NM_001351800.1:c.3071G > A:p.Arg1024Gln), respectively. In addition, we identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of",220,1552,35974224f8ab2ccb8e3e9195323ab8c5e074253a0cb2e3859b7418b65430a896,section_word_window_220_overlap_40_v3,460d5cd5-95d3-495d-ba67-21bdceb792c2,fcfabe4d-a5c0-4cf9-bcaa-b347cd7b05d6,2026-09-13T22:10:47.392Z,2026-09-13T22:10:47.392Z
PMC13538227,PMC13538227.1,1,1,0,abstract,null,abstract,Abstract,Abstract,0,identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of CeAD. Conclusion Systematic search for rare disease-causing variants should not be recommended in all sCeAD cases but it should be limited to selected individuals with a high pre-test probability to harbor a monogenic disease. Supplementary Information The online version contains supplementary material available at https://doi.org/10.1007/s10072-026-09308-6.,85,619,ec255bf85949fb56f84bdbe31bfdde27975075c7a32748a69192702eb2ce34bf,section_word_window_220_overlap_40_v3,460d5cd5-95d3-495d-ba67-21bdceb792c2,fcfabe4d-a5c0-4cf9-bcaa-b347cd7b05d6,2026-09-13T22:10:47.392Z,2026-09-13T22:10:47.392Z
PMC13538227,PMC13538227.1,2,0,1,Sec1,null,section,Introduction,Introduction,1,"Arterial dissection is, by definition, the accumulation of blood within the wall of an artery. Once considered uncommon, dissections of carotid or vertebral arteries (cervical artery dissections, CeADs) are now recognized as the major cause of ischemic stroke in young adults, accounting for approximately 20% of cases in patients under 45 years of age. Notwithstanding, the pathogenesis of CeAD is still poorly defined, especially in those cases that occur spontaneously (spontaneous CeAD, sCeAD), without any identifiable precipitating events [ 1 ]. Several arguments point toward a key role played by the connective tissue component of the arterial wall. The finding of composite collagen fibrils and fragmented elastic fibers on electron microscopic examination of skin biopsy specimens in more than half of patients with sCeAD [ 2 – 4 ] and the id

pmcid,article_version,chunking_status,chunk_count,error_message,updated_at
PMC13538227,PMC13538227.1,completed,25,null,2026-09-13T22:10:47.392Z
PMC13538279,PMC13538279.1,completed,34,null,2026-09-13T22:10:47.392Z
PMC13539412,PMC13539412.1,completed,43,null,2026-09-13T22:10:47.393Z
PMC13540131,PMC13540131.2,completed,35,null,2026-09-13T22:10:47.394Z
PMC13543812,PMC13543812.1,completed,38,null,2026-09-13T22:10:47.395Z
PMC13544149,PMC13544149.1,completed,21,null,2026-09-13T22:10:47.396Z
PMC13545448,PMC13545448.1,completed,48,null,2026-09-13T22:10:47.396Z
PMC13547081,PMC13547081.1,completed,14,null,2026-09-13T22:10:47.397Z
PMC13547469,PMC13547469.1,completed,24,null,2026-09-13T22:10:47.398Z
PMC13552502,PMC13552502.1,completed,48,null,2026-09-13T22:10:47.398Z


In [0]:

# ============================================================
# Validaciones de calidad
# ============================================================

if chunks_df is None:
    raise RuntimeError(
        "No chunks were generated."
    )

quality_df = (
    chunks_df
    .agg(
        F.count("*").alias(
            "total_chunks"
        ),

        F.countDistinct(
            F.struct(
                "pmcid",
                "article_version",
                "chunk_index",
            )
        ).alias(
            "distinct_chunk_keys"
        ),

        F.countDistinct(
            "pmcid"
        ).alias(
            "documents_with_chunks"
        ),

        F.countDistinct(
            F.struct(
                "pmcid",
                "article_version",
                "section_index",
            )
        ).alias(
            "sections_with_chunks"
        ),

        F.round(
            F.avg(
                "word_count"
            ),
            2,
        ).alias(
            "avg_words_per_chunk"
        ),

        F.min(
            "word_count"
        ).alias(
            "min_words_per_chunk"
        ),

        F.max(
            "word_count"
        ).alias(
            "max_words_per_chunk"
        ),

        F.sum(
            F.when(
                F.lower(
                    F.coalesce(
                        F.col(
                            "section_title"
                        ),
                        F.lit(""),
                    )
                ).rlike(
                    "^(references?|bibliography)"
                ),
                1,
            ).otherwise(0)
        ).alias(
            "reference_chunks"
        ),

        F.countDistinct(
            "content_hash"
        ).alias(
            "distinct_content_hashes"
        ),
    )
)

quality = quality_df.first()

if (
    quality["total_chunks"]
    != quality[
        "distinct_chunk_keys"
    ]
):
    raise RuntimeError(
        "Duplicate chunk keys detected."
    )

if quality[
    "reference_chunks"
] != 0:
    raise RuntimeError(
        "Reference chunks were generated unexpectedly."
    )

display(
    quality_df
)

display(
    chunks_df
    .groupBy(
        "section_title"
    )
    .agg(
        F.count("*").alias(
            "chunks"
        )
    )
    .orderBy(
        F.desc(
            "chunks"
        )
    )
    .limit(30)
)


total_chunks,distinct_chunk_keys,documents_with_chunks,sections_with_chunks,avg_words_per_chunk,min_words_per_chunk,max_words_per_chunk,reference_chunks,distinct_content_hashes
643,643,20,324,167.92,20,220,0,627


section_title,chunks
Discussion,63
Introduction,44
Abstract,30
Results,19
Body,10
Multiple Sclerosis and P2X7R,9
Case Presentation,8
AI ranking efficiency and pitfall among ES analysis,7
Conclusion,7
National and international databases,7


In [0]:

# ============================================================
# Persistencia autoritativa por documento
# ============================================================

processed_document_keys_df = (
    pending_documents_df
    .select(
        "pmcid",
        "article_version",
    )
    .distinct()
)

chunk_delta = DeltaTable.forName(
    spark,
    CHUNK_TABLE,
)

# Eliminar cualquier versión anterior de los chunks de los documentos
# procesados. Así no quedan filas obsoletas si una nueva ejecución
# produce menos fragmentos.
(
    chunk_delta.alias("target")
    .merge(
        processed_document_keys_df.alias("source"),
        '''
        target.pmcid = source.pmcid
        AND target.article_version = source.article_version
        ''',
    )
    .whenMatchedDelete()
    .execute()
)

if chunks_df is not None:
    (
        chunks_df
        .write
        .mode("append")
        .saveAsTable(CHUNK_TABLE)
    )

print(
    "Chunks replaced for processed documents."
)


Chunks replaced for processed documents.


In [0]:
# ============================================================
# Actualizar chunking_status en pmc_inventory
# ============================================================
#
# Usamos document_status_df construido en Python, por lo que no depende
# de reevaluar pending_documents_df después de modificar la tabla.
# Esto evita el problema de lazy evaluation visto en el script 04 y
# funciona en Databricks Serverless.

inventory_delta = DeltaTable.forName(
    spark,
    INVENTORY_TABLE,
)

(
    inventory_delta.alias("target")
    .merge(
        document_status_df.alias("source"),
        '''
        target.pmcid = source.pmcid
        AND target.article_version = source.article_version
        ''',
    )
    .whenMatchedUpdate(
        set={
            "chunking_status":
                "source.chunking_status",

            "error_message":
                "source.error_message",

            "updated_at":
                "source.updated_at",
        }
    )
    .execute()
)

print(
    "pmc_inventory chunking statuses updated."
)

pmc_inventory chunking statuses updated.


In [0]:
# ============================================================
# Registrar ejecución
# ============================================================

RUN_COMPLETED_AT = datetime.now(
    timezone.utc
)

# Leer métricas desde la tabla ya persistida usando el RUN_ID.
# De esta forma la auditoría también es serverless-safe.

persisted_run_chunks_df = (
    spark.table(CHUNK_TABLE)
    .filter(
        F.col(
            "chunking_run_id"
        ) == RUN_ID
    )
)

records_processed = len(
    document_status_results
)

records_failed = sum(
    record[
        "chunking_status"
    ] == "failed"
    for record in document_status_results
)

records_successful = (
    records_processed
    - records_failed
)

persisted_chunk_count = (
    persisted_run_chunks_df.count()
)

run_row = spark.createDataFrame(
    [
        (
            RUN_ID,
            PIPELINE_NAME,
            (
                "completed"
                if records_failed == 0
                else "completed_with_errors"
            ),
            RUN_STARTED_AT,
            RUN_COMPLETED_AT,
            int(MAX_DOCUMENTS),
            int(documents_selected),
            int(records_processed),
            int(records_successful),
            0,
            int(records_failed),
            json.dumps({
                "source_table":
                    CLEAN_SECTION_TABLE,

                "chunk_table":
                    CHUNK_TABLE,

                "chunking_method":
                    CHUNKING_METHOD,

                "chunk_size_words":
                    CHUNK_SIZE_WORDS,

                "chunk_overlap_words":
                    CHUNK_OVERLAP_WORDS,

                "minimum_section_words":
                    MIN_SECTION_WORDS,

                "chunks_generated":
                    persisted_chunk_count,

                "section_aware":
                    True,

                "references_excluded":
                    True,

                "authoritative_replace":
                    True,

                "content_hash_enabled":
                    True,

                "document_metadata_source":
                    INVENTORY_TABLE,

                "serverless_safe":
                    True,
            }),
            None,
        )
    ],
    schema=T.StructType([
        T.StructField(
            "run_id",
            T.StringType(),
            False,
        ),
        T.StructField(
            "pipeline_name",
            T.StringType(),
            False,
        ),
        T.StructField(
            "run_status",
            T.StringType(),
            False,
        ),
        T.StructField(
            "started_at",
            T.TimestampType(),
            False,
        ),
        T.StructField(
            "completed_at",
            T.TimestampType(),
            True,
        ),
        T.StructField(
            "records_requested",
            T.LongType(),
            True,
        ),
        T.StructField(
            "records_found",
            T.LongType(),
            True,
        ),
        T.StructField(
            "records_processed",
            T.LongType(),
            True,
        ),
        T.StructField(
            "records_inserted",
            T.LongType(),
            True,
        ),
        T.StructField(
            "records_updated",
            T.LongType(),
            True,
        ),
        T.StructField(
            "records_failed",
            T.LongType(),
            True,
        ),
        T.StructField(
            "execution_metadata",
            T.StringType(),
            True,
        ),
        T.StructField(
            "error_message",
            T.StringType(),
            True,
        ),
    ]),
)

run_row.write.mode(
    "append"
).saveAsTable(
    PIPELINE_RUNS_TABLE
)

print(
    f"Pipeline run registered: {RUN_ID}"
)

Pipeline run registered: fcfabe4d-a5c0-4cf9-bcaa-b347cd7b05d6


In [0]:

# ============================================================
# Auditoría final
# ============================================================

final_chunks_df = (
    spark.table(CHUNK_TABLE)
    .filter(
        F.col(
            "chunking_run_id"
        ) == RUN_ID
    )
)

print("===== INVENTORY =====")
display(
    spark.table(INVENTORY_TABLE)
    .filter(
        F.col("is_latest_version") == True
    )
    .select(
        "pmcid",
        "article_version",
        "cleaning_status",
        "chunking_status",
        "error_message",
    )
    .orderBy(
        "pmcid",
        "article_version",
    )
)

print("===== CHUNK SUMMARY =====")
display(
    final_chunks_df
    .agg(
        F.count("*").alias(
            "chunks"
        ),

        F.countDistinct(
            "pmcid"
        ).alias(
            "documents"
        ),

        F.countDistinct(
            F.struct(
                "pmcid",
                "article_version",
                "section_index",
            )
        ).alias(
            "sections"
        ),

        F.round(
            F.avg(
                "word_count"
            ),
            2,
        ).alias(
            "avg_words"
        ),

        F.min(
            "word_count"
        ).alias(
            "min_words"
        ),

        F.max(
            "word_count"
        ).alias(
            "max_words"
        ),
    )
)

print("===== CHUNKS - CURRENT RUN =====")
display(
    final_chunks_df
    .select(
        "pmcid",
        "article_version",

        "chunk_index",
        "section_chunk_index",

        "section_title",
        "section_path",

        "word_count",
        "character_count",
        "content_hash",

        "chunk_text",
    )
    .orderBy(
        "pmcid",
        "article_version",
        "chunk_index",
    )
)

print("===== PIPELINE RUN =====")
display(
    spark.table(PIPELINE_RUNS_TABLE)
    .filter(
        F.col("run_id") == RUN_ID
    )
)


===== INVENTORY =====


pmcid,article_version,cleaning_status,chunking_status,error_message
PMC13538227,PMC13538227.1,completed,completed,null
PMC13538279,PMC13538279.1,completed,completed,null
PMC13539412,PMC13539412.1,completed,completed,null
PMC13540131,PMC13540131.2,completed,completed,null
PMC13543812,PMC13543812.1,completed,completed,null
PMC13544149,PMC13544149.1,completed,completed,null
PMC13545448,PMC13545448.1,completed,completed,null
PMC13547081,PMC13547081.1,completed,completed,null
PMC13547469,PMC13547469.1,completed,completed,null
PMC13552502,PMC13552502.1,completed,completed,null


===== CHUNK SUMMARY =====


chunks,documents,sections,avg_words,min_words,max_words
643,20,324,167.92,20,220


===== CHUNKS - CURRENT RUN =====


pmcid,article_version,chunk_index,section_chunk_index,section_title,section_path,word_count,character_count,content_hash,chunk_text
PMC13538227,PMC13538227.1,0,0,Abstract,Abstract,220,1552,35974224f8ab2ccb8e3e9195323ab8c5e074253a0cb2e3859b7418b65430a896,"Introduction Whether spontaneous cervical artery dissection (sCeAD), the leading cause of ischemic stroke in young adults, represents the manifestation of unrecognized hereditary connective tissue disorders (HCTDs) and whether HCTDs have a major impact in the epidemiology of the disease is a matter of ongoing debate. We aimed at determining the frequency of clinically relevant genetic variants (CRGVs) in a cohort of unselected sCeAD patients by targeted next-generation sequencing (NGS) approach. Methods We designed a high-throughput sequencing panel to identify variants in 38 candidate genes associated with arterial dissection or aneurysm and screened patients with apparently sporadic sCeAD, consecutively referred to one comprehensive stroke center from August 2020 to December 2025. The frequency of known disease-causing and pertinent variants of uncertain significance (VUS) was calculated. Then, we performed a systematic review of all studies evaluating the prevalence of monogenic disorders among sCeAD patients up to December 2025. Results Among 183 patients (males, 51.3%; mean age, 42.0 ± 11.3 years), 2 (1.1%) carried a CeAD-causing variant in COL3A1 ( NM_000090.4:c.2959G > A:p.Gly987Ser) and ABCC6 ( NM_001351800.1:c.3071G > A:p.Arg1024Gln), respectively. In addition, we identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of"
PMC13538227,PMC13538227.1,1,1,Abstract,Abstract,85,619,ec255bf85949fb56f84bdbe31bfdde27975075c7a32748a69192702eb2ce34bf,identified 30 (16.4%) VUS in 25 (13.6%) patients. The analysis of 330 patients across 14 studies yielded a prevalence of carriers of disease-causing variants ranging between 0.4% in series of unselected patients and 23.4% in patients with familial history of CeAD. Conclusion Systematic search for rare disease-causing variants should not be recommended in all sCeAD cases but it should be limited to selected individuals with a high pre-test probability to harbor a monogenic disease. Supplementary Information The online version contains supplementary material available at https://doi.org/10.1007/s10072-026-09308-6.
PMC13538227,PMC13538227.1,2,0,Introduction,Introduction,220,1466,4ecdc0218a38e24945d5b9a395e5fca9eb94eb1495de02d9b8de3e97d7249779,"Arterial dissection is, by definition, the accumulation of blood within the wall of an artery. Once considered uncommon, dissections of carotid or vertebral arteries (cervical artery dissections, CeADs) are now recognized as the major cause of ischemic stroke in young adults, accounting for approximately 20% of cases in patients under 45 years of age. Notwithstanding, the pathogenesis of CeAD is still poorly defined, especially in those cases that occur spontaneously (spontaneous CeAD, sCeAD), without any identifiable precipitating events [ 1 ]. Several arguments point toward a key role played by the connective tissue component of the arterial wall. The finding of composite collagen fibrils and fragmented elastic fibers on electron microscopic examination of skin biopsy specimens in more than half of patients with sCeAD [ 2 – 4 ] and the identification of clinically detectable signs of connective tissue aberration in most sCeAD patients support the hypothesis of a generalized structural connective tissue defect predisposing to disease occurrence, even in sporadic cases without evident signs of a known hereditary connective tissue disorder (HCTD) [ 5, 6 ]. In this view, the reduced or altered biosynthesis of the extracellular matrix (ECM) components might lead to a generalized structural weakness of 

===== PIPELINE RUN =====


run_id,pipeline_name,run_status,started_at,completed_at,records_requested,records_found,records_processed,records_inserted,records_updated,records_failed,execution_metadata,error_message
fcfabe4d-a5c0-4cf9-bcaa-b347cd7b05d6,05_Create_Document_Chunks_v3_Clean,completed,2026-09-13T22:10:26.580Z,2026-09-13T22:11:04.762Z,20,20,20,20,0,0,"{""source_table"": ""workspace.tfm_pmc.pmc_clean_sections"", ""chunk_table"": ""workspace.tfm_pmc.pmc_document_chunks"", ""chunking_method"": ""section_word_window_220_overlap_40_v3"", ""chunk_size_words"": 220, ""chunk_overlap_words"": 40, ""minimum_section_words"": 20, ""chunks_generated"": 643, ""section_aware"": true, ""references_excluded"": true, ""authoritative_replace"": true, ""content_hash_enabled"": true, ""document_metadata_source"": ""workspace.tfm_pmc.pmc_inventory"", ""serverless_safe"": true}",null
